# "Surplus Exergy" 

NOTE: _x_ and _g_ are used interchangeably in this notebook, both represents ore-grade

## Preparations 
* Libraries used
* Functions
* Establishing the BrightWay2.5 environment <br>
**Note**: Maybe you want to modify the 3rd code block here to assign the whole computation process into a specific project

In [39]:
import numpy as np
import pandas as pd

# =============================================
# Core Scientific/LCA (Life Cycle Assessment) Libraries
# Required for Brightway2.5 (LCA framework)
# Always place these at the top, as they are the main dependencies for your project
# =============================================
import os
import bw2io as bi # Brightway2 library for data import/export
import bw2data as bd # Core Brightway2 library for LCA database management
import bw2calc as bc

# =============================================
# Data Processing and Numerical Libraries
# Used for data manipulation, math, and array operations
# =============================================
import numpy as np  # Numerical computing (arrays, math operations), only version for np.nan # numpy==1.26.4
import pandas as pd # Data manipulation and analysis (DataFrames)
import math         # Basic math functions
import csv          # CSV file reading/writing
import os           # Operating system interfaces (file paths, environment variables)
import re           # Regular expressions (text pattern matching)

# =============================================
# File System and Path Handling
# Used for file and directory operations
# =============================================
from pathlib import Path  # Object-oriented filesystem paths

# =============================================
# Time and Delay Utilities
# Used for introducing delays or measuring time
# =============================================
from time import sleep  # Introduce delays (e.g., for web scraping)
import time             # Time-related functions (e.g., `time.time()`)

# =============================================
# Chemistry and Data Retrieval
# Used for chemical data and external API access
# =============================================
import pubchempy as pcp                 # Interface to PubChem (chemical data)
from pubchempy import PubChemHTTPError  # Unique CF assignment for each flow that contains copper
import requests                         # HTTP requests (e.g., API calls)
from mendeleev import element           # Periodic table data (element properties)

In [40]:
# 1. The name of each flow will be cleaned using re
def clean_chemical_name(name):
    # This detects some of the organic molecules (e.g. Ethane, 1,1-difluoro-, HFC-152a)
    pattern1 = r"^(.+?),\s+(.+?)-(?:,\s+.*|$)"
    # This pattern simple detects the flows with a comma.
    pattern2 = r"(?<=[a-zA-Z]),.*"

    # First, check if the flow satisfies pattern1
    match = re.search(pattern1,name)
    # If there is a match, then we change its format (e.g. from Ethane, 1,1-difluoro-, HFC-152a to 1,1-difluoroEthane)
    if match:
        name = re.sub(pattern1, r"\g<2>\g<1>", name)
    # If not, we apply the 2nd pattern
    else:
        name = re.sub(pattern2, "", name)
    
    # Second, remove Roman numerals at the end (e.g., Aluminium III -> Aluminium) if applicable
    name = re.sub(r"\s+[IVXLCDM]+$", "", name)
    
    # 3. Third the word 'ion' or 'ions' at the end (e.g., Copper ion -> Copper) if applicable
    # Using re.IGNORECASE makes it catch 'Ion', 'ION', or 'ion'
    name = re.sub(r"\s+ions?$", "", name, flags=re.IGNORECASE)
    
    return name.strip()

# 2. Then, it will be fed to the API to find the chemical. If it is found, the original flow name, the chemical IUPAC name, molecular composition, and the molecular weight will be made into a dictionary
def chem_comp(mf):
    # First of all, we need to use re to cut the molecular formula into pieces
    # Regex breakdown:
    # ([A-Z][a-z]?) -> Matches an Uppercase letter potentially followed by a lowercase (the element)
    # (\d*)         -> Matches zero or more digits following the element (the count)
    pattern = r"([A-Z][a-z]?)(\d*)"
    
    matches = re.findall(pattern, mf)
    
    parts = []

    for elem,count in matches:
        # If the count is empty (like in 'Cl'), it means there is 1 atom
        count = int(count) if count else 1
        parts.append([elem,count])
    
    return parts

# Added here a dict input to switch between CF1 and CF2 dictionaries
def has_elem(parts,dict): 
    for elem, count in parts:
        if elem not in dict.keys(): #if it is not included in the list of elements
            print(f"{elem} is not included in the scope.")
            continue
        else:
            return True
    return None

# 3. Calculates for the CF of the given molecular formular
# Added here a dict input to switch between CF1 and CF2 dictionaries
def cf_calculator(dict,mw,parts):
    cf=0 # place holder for the 
    for elem, count in parts:
        if elem in dict.keys():
            # Find the atomic mass of the element first
            el_data = element(elem) # from mendeley library
            mass = float(el_data.atomic_weight) # from mendeley library
            # Compute for the contribution of this element in the final mf
            cf += count*(mass/mw)*dict[elem]
        else:
            continue
    return cf

In [41]:
# Set up the project and database
project_name = "Ecoinvent3.4" # CHANGE THE PROJECT NAME IF YOU WISH
bd.projects.set_current(project_name)

# Retrieves credentials that are stored in the os.
username = os.getenv("EI_USERNAME")
password = os.getenv("EI_PASSWORD")

# Use this code if the project is not imported yet
#bi.import_ecoinvent_release("3.4", "cutoff", username, password)

# 1. Initialize the database object
biosphere = bd.Database('ecoinvent-3.4-biosphere')

bd.databases

Databases dictionary with 2 object(s):
	ecoinvent-3.4-biosphere
	ecoinvent-3.4-cutoff

## $CF_1$ Calculation

$CF_1 = \frac{\partial g}{\partial CMT} = -\frac{A \beta e^{\alpha}}{CMT^2}(\frac{A}{CMT}-1)^{\beta - 1}$

### 1.1 Import data and compute for CF1 of each target element
Note: CF for gold cannot be computed because its amount extracted is greater than the remaining reserve. This gives a mathematical error when doing <br>
$(\frac{A}{CMT}-1)^{\beta-1}$

In [42]:
# Import the data from local excel file
OGD_df = pd.read_excel("Ore-GradeDeclineConstants.xlsx")

# Grouping the numerator and denominator explicitly
numerator = OGD_df['URR'] * OGD_df['beta'] * np.exp(OGD_df['alpha'])
denominator = OGD_df['CME']**2.0 #Forced into a float computation because the number is too large to handle

# Explicitly separated calculation
OGD_df['CF1'] = -(numerator / denominator) * ((OGD_df['URR'] / OGD_df['CME']) - 1)**(OGD_df['beta'] - 1)

# Removing gold from the list. See the reason listed in the markdown file
OGD_df = OGD_df[OGD_df['Metal']!= "Gold"]
OGD_df

,Metal,Symbol,alpha,beta,URR,CME,k,CF1
0,Aluminium,Al,-1.35,0.10,13400000000000000,1040000000000,1041.0,-6.422537e-14
1,Antimony,Sb,-2.06,0.42,66100000000,6790000000,40.0,-2.183408e-11
2,Chromium,Cr,-1.15,0.12,15200000000000,206000000000,18.0,-3.127861e-13
3,Cobalt,Co,-4.86,0.17,2860000000000,2280000000,NaN,-1.944750e-12
4,Copper,Cu,-3.61,0.17,4360000000000,592000000000,170.0,-1.231243e-14
6,Iron,Fe,-0.57,0.13,6460000000000000,34100000000000,78.0,-4.282654e-15
7,Lead,Pb,-2.61,0.21,2810000000000,235000000000,21.0,-1.185525e-13
8,Lithium,Li,-4.95,0.11,3470000000000,9810000000,88.0,-1.518493e-13
9,Manganese,Mn,-1.19,0.08,127000000000000,580000000000,8.0,-6.485008e-14
10,Molybdenum,Mo,-6.43,0.27,182000000000,6620000000,660.0,-1.653204e-13


### 1.2 Construct a CF1_dict and assign unique CF1 to each INPUT flows of the database
This step is expected to take about 5 mins. It is much shorter than the previous project because we are only targeting the input flows.

In [46]:
# Build a data dictionary with the element symbols as the keys
CF1_dict = dict(zip(OGD_df.iloc[:,1],OGD_df.iloc[:,7]))
CF1_dict # have a look how it looks like

{'Al': -6.422536676269253e-14,
 'Sb': -2.1834077210692155e-11,
 'Cr': -3.1278613038877847e-13,
 'Co': -1.944749514845395e-12,
 'Cu': -1.2312433333350088e-14,
 'Fe': -4.282654305987987e-15,
 'Pb': -1.1855254708054443e-13,
 'Li': -1.518493452995773e-13,
 'Mn': -6.485008032108111e-14,
 'Mo': -1.653204348388365e-13,
 'Ni': -9.066741688482605e-14,
 'Nb': -1.6298072211886446e-11,
 'P': -1.6973366253890836e-14,
 'Ag': -1.5701960660375504e-13,
 'Sn': -1.3268575464212294e-13,
 'U': -9.154438878304105e-12,
 'Zn': -1.083657674571144e-13}

In [47]:
# we need a if-else filter here to see if the flow is elementary. If yes, do not go through the cf_calculator

method_data = []
extra_info = []

not_found = []
not_emission = [] # can be used to check for the flows that aren't considered

i=0
for flow in biosphere:
    cf = None
    parts = None #reset parts every time
    # Only consider the elementary flows going into the biosphere
    # Find flows that contain any of the search terms
    if isinstance(flow.get('categories'), tuple) and len(flow['categories']) > 0 and flow['unit'].lower() == 'kilogram' and flow['categories'][0].lower() == 'natural resource':

        flow_name_cleaned = clean_chemical_name(flow['name'])

        sleep(0.3) #Slow down the request to the API to avoid sudden shut down of connection

        max_retries = 3
        for attempt in range(max_retries):
            try:
                compound = pcp.get_compounds(flow_name_cleaned,'name')
                if compound: 
                    c = compound[0]
                    mf = c.molecular_formula
                    mw = c.molecular_weight
            
                    parts = chem_comp(mf)

                    # If the flow contains only one element, we can directly use its CF value
                    if len(parts)==1:
                        if has_elem(parts,CF1_dict): #if the only element is a part of the metal group
                            cf = CF1_dict.get(parts[0][0])   
                            print(f"The cf of {mf} is decided to be {cf}")
                        else:
                            print(f"{flow['name']} does not contain any element of the scope.")       
                    # If the flow contains a compound, use the cf_calculator
                    elif len(parts)>1:
                        if has_elem(parts,CF1_dict):
                            cf = cf_calculator(CF1_dict,mw,parts)
                            print(f"The cf of {mf} is calculated to be {cf}")
                        else:
                            print(f"{flow['name']} does not contain any element of the scope.")   
                    else: 
                        print(f"The molecular compound does not contain any element.")
                    
                    # Combine the original flow name, 
                    if cf is not None:
                        method_data.append((flow.key,float(cf)))
                        extra_info.append([flow.key,flow['name'],flow_name_cleaned,mf,mw,float(cf)])
                    else: 
                        extra_info.append([flow.key,flow['name'],flow_name_cleaned,mf,mw,None])
                        print('This compound does not contain any element of the scope.')
                else:
                    print(f"Compound not found for: {flow['name']}")
                    not_found.append([flow.key,flow['name']])
                break

            except PubChemHTTPError as e:
                # Check if it's a 502 or another server error
                if "502" in str(e) and attempt < max_retries - 1:
                    print(f"Server hit a 502 for {flow_name_cleaned}. Retrying in 2 seconds... (Attempt {attempt + 1}/{max_retries})")
                    time.sleep(2)  # Give the server a moment to recover
                else:
                    print(f"Failed to fetch {flow_name_cleaned} after multiple attempts or met a different error: {e}")
                    # Handle or log the permanent failure here
                    break
    else:
        not_emission.append([flow.key,flow['name']])


Au is not included in the scope.
Gold, Au 2.1E-4%, Ag 2.1E-4%, in ore, in ground does not contain any element of the scope.
This compound does not contain any element of the scope.
The cf of Cu is decided to be -1.2312433333350088e-14
The cf of Pb is decided to be -1.1855254708054443e-13
The cf of Ag is decided to be -1.5701960660375504e-13
Hg is not included in the scope.
S is not included in the scope.
Cinnabar, in ground does not contain any element of the scope.
This compound does not contain any element of the scope.
Sr is not included in the scope.
strontium, in ground does not contain any element of the scope.
This compound does not contain any element of the scope.
C is not included in the scope.
Ca is not included in the scope.
O is not included in the scope.
Calcite, in ground does not contain any element of the scope.
This compound does not contain any element of the scope.
Pd is not included in the scope.
Pd, Pd 7.3E-4%, Pt 2.5E-4%, Rh 2.0E-5%, Ni 2.3E+0%, Cu 3.2E+0% in ore

In [ ]:
# Activate this block of code if you want to track the details of the previous step

#PC_df = pd.DataFrame(extra_info)
#NF_df = pd.DataFrame(not_found)
#NInc_df = pd.DataFrame(not_emission)
#method_df = pd.DataFrame(method_data)
#PC_df.to_csv("data/PubChem_Info_EI34.csv", index=False)
#NF_df.to_csv("data/PubChem_NotFound_EI34.csv", index=False)
#NInc_df.to_csv("data/PubChem_NotEmission_EI34.csv", index=False)
#method_df.to_csv("data/PubChem_Method_EI34.csv", index=False)

### 1.3 Upload this intermediate LCIA method to the project file in BrightWay2.5
You may check if I put the right description to the method file

In [48]:
# 1. Define the method name as a tuple for the desired hierarchy
method_name_tuple = (
    "Future Effort method",
    "Ore-Grade Decline Potential",
    "Applied to 17 elements"  # Updated name
)

# 2. Define metadata for the method, including the unit and a description
method_metadata = {
    'unit': 'change in ore grade per kg of metal extracted',
    'description': 'This LCIA method is one out of the 3 steps of the currently developing surplus exergy method.'
                    'It models the decrease of ore-grade with the progression of the extraction activities.'
                    'The impact score (IS) isn\'t the final value of the intended impact yet. '
                    'This IS needs to be fed to two more calculations to find out the exergy lost due to the dissipation in the product system.',
    'source': 'The values are taken from the appendix of ReCiPe 2016: https://www.rivm.nl/bibliotheek/rapporten/2016-0104.pdf',
    'version': '2.0',
    'num_cfs': len(method_data),
    'application': 'Input product-system metals characterization'
}

# 3. Create or load the Brightway2 Method object
method_object = bd.Method(method_name_tuple)

# 4. Forcefully register/write the method
try:
    # This will overwrite existing method
    method_object.register(**method_metadata) 
    method_object.write(method_data)
    
    print(f"\n✅ Successfully {'overwrote' if method_name_tuple in bd.methods else 'created'} method: {method_name_tuple}")
    print(f"   - Unit: {method_metadata['unit']}")
    print(f"   - Number of CFs: {len(method_data)}")
    
except Exception as e:
    print(f"\n❌ Failed to write method: {str(e)}")
    raise


# --- Verification ---
print("\n--- Verification ---")
if method_name_tuple in bd.methods:
    retrieved_method = bd.Method(method_name_tuple)
    loaded_data = retrieved_method.load()
    
    print(f"🔍 Method verification:")
    print(f"   Name: {retrieved_method.name}")
    print(f"   Metadata version: {retrieved_method.metadata.get('version', 'N/A')}")
    print(f"   Number of CFs loaded: {len(loaded_data)}")
    
    # Check for potential data loss
    if len(loaded_data) != len(method_data):
        print(f"⚠️  Warning: CF count mismatch. Expected {len(method_data)}, got {len(loaded_data)}")
    else:
        print("✅ CF count matches expected value")
        
    # Show sample CFs
    print("\nSample characterization factors (first 3):")
    for cf in loaded_data[:3]:
        flow = bd.get_activity(cf[0])
        print(f"   - {flow['name']}: {cf[1]}")
else:
    print(f"❌ Error: Method {method_name_tuple} not found after writing attempt")

print("\nProcess completed")


✅ Successfully overwrote method: ('Future Effort method', 'Ore-Grade Decline Potential', 'Applied to 17 elements')
   - Unit: change in ore grade per kg of metal extracted
   - Number of CFs: 66

--- Verification ---
🔍 Method verification:
   Name: ('Future Effort method', 'Ore-Grade Decline Potential', 'Applied to 17 elements')
   Metadata version: 2.0
   Number of CFs loaded: 66
✅ CF count matches expected value

Sample characterization factors (first 3):
   - Copper, 0.99% in sulfide, Cu 0.36% and Mo 8.2E-3% in crude ore, in ground: -1.2312433333350088e-14
   - Lead, 5.0% in sulfide, Pb 3.0%, Zn, Ag, Cd, In, in ground: -1.1855254708054443e-13
   - Silver, Ag 1.5E-5%, Au 5.4E-4%, in ore, in ground: -1.5701960660375504e-13

Process completed


## $CF_2$ Calculation (Unique to each analysis)
#### The flow chosen is 500,000 kg of market for copper cathode
1. Download data of the elemental contribution from the first LCIA.
2. Calculate the corresponding decrease in ore grade for each element associated in the process. <br>
_ImpactScore_1_ = $\Delta x$; <br>
Computing for the initial ore-grade:
$x_i=e^{\alpha}(\frac{A}{T}-1)^{\beta}$ <br>
$x_f=x_i-\Delta x$

### Option 1
3. Calculate the change in concentration exergy, $\Delta b_c$, based on the change in ore-grade, $\Delta x$ <br>
$b_c(x)=-RTº[ln(x)+(1-x)ln(1-x)]$ <br>
$\Delta b_c = CF_2 = |{b_c(x_i-\Delta g)-b_c{x_i}}|$, unit: kJ <br>
R = 8.314 kJ/kmol•K <br>
Tº = 290.15K = 17ºC (This is weird, but I will follow what is given by Vieira et al.)

### Option 2
3. Calculate the change in ERC, $k\Delta b_c$, based on the change in ore-grade, $\Delta x$ <br>
$b_c(x)=-RTº[ln(x)+(1-x)ln(1-x)]$ <br>
$ERC = k\Delta b_c = CF_2 = k|{b_c(x_i-\Delta g)-b_c{x_i}}|$, unit: kJ <br>
R = 8.314 kJ/kmol•K <br>
Tº = 290.15K = 17ºC (This is weird, but I will follow what is given by Vieira et al.)

### 2.1 Ore-grade variation of copper production

In [59]:
# Import data 
IS1_df = pd.read_csv('manual4CuCathode/IS4CF1_2.csv')

# Data cleaning, keeping only the flow name and the impact score, which represents the ore-grade decrease
IS1_df_cleaned = IS1_df.drop(columns = ['Unnamed: 0','index','categories','type','unit','database'])
IS1_df_cleaned = IS1_df_cleaned.iloc[2:]
IS1_df_cleaned.columns = ['flow name','OG decrease']
IS1_df_cleaned

,flow name,OG decrease
2,"Chromium, 25.5% in chromite, 11.6% in crude or...",-2.061579e-09
3,"Clay, unspecified, in ground",-1.559967e-09
4,"Zinc, Zn 0.63%, in mixed ore, in ground",-1.365437e-09
5,"Lead, Pb 0.014%, in mixed ore, in ground",-1.152349e-09
6,"Copper, 0.99% in sulfide, Cu 0.36% and Mo 8.2E...",-1.090108e-09
7,"Copper, Cu 0.38%, in mixed ore, in ground",-9.902189e-10
8,"Copper, 0.52% in sulfide, Cu 0.27% and Mo 8.2E...",-7.391825e-10
9,"Copper, 1.18% in sulfide, Cu 0.39% and Mo 8.2E...",-5.833958e-10
10,"Nickel, 1.13% in sulfide, Ni 0.76% and Cu 0.76...",-5.712201e-10
11,"Aluminium, in ground",-4.825921e-10


In [60]:
# Generalize the flows by looking at just the metal the flow is related to
for index, row in IS1_df_cleaned.iterrows():
    # RE used to capture the text before the first comma
    # Feed it with the name of the flow, it will return the type of resources
    match = re.match(r"^[^,]+", row['flow name'])
    IS1_df_cleaned.at[index, 'flow name'] = match[0]

# Remove clay, because I think this is not counted as any ore deposit. It is included because PubChem gives a chemical formula of Al2O5Si to it.
IS1_df_cleaned = IS1_df_cleaned.drop(IS1_df_cleaned.index[1])

IS1_df_cleaned


,flow name,OG decrease
2,Chromium,-2.061579e-09
4,Zinc,-1.365437e-09
5,Lead,-1.152349e-09
6,Copper,-1.090108e-09
7,Copper,-9.902189e-10
8,Copper,-7.391825e-10
9,Copper,-5.833958e-10
10,Nickel,-5.712201e-10
11,Aluminium,-4.825921e-10
12,Copper,-4.108229e-10


In [61]:
# Need to manually modify 1 flows (Ni) because of the inconsistency in its name
IS1_df_cleaned.at[19,'flow name'] = 'Nickel'

# Group by the element each flow represents and sum up the impact values
combined_IS_df = IS1_df_cleaned.groupby('flow name', as_index=False)['OG decrease'].sum()
combined_IS_df

,flow name,OG decrease
0,Aluminium,-4.825921e-10
1,Chromium,-2.061579e-09
2,Copper,-4.125942e-09
3,Iron,-1.820008e-10
4,Lead,-1.391878e-09
5,Molybdenum,-5.935842e-10
6,Nickel,-1.098175e-09
7,Zinc,-1.759550e-09


### 2.2 Calculation of the change in concentration exergy of each ore

In [62]:
# Define two unique functions for this step
# OG_ini calculates the theoretical initial mine concentration based on the ore-grade decline model
def OG_ini(alpha,beta,URR,CME):
    xi = np.exp(alpha)*((URR/CME)-1)**beta
    return xi
# Delta_bc calculates the change in concentration exergy that corresponds to the change in the change in ore grade
def Delta_bc(xi,dg): #dg here is expected to be negative
    R = 8.314
    T = 290.15 #Defined by Vieira et al., even though this is normally taken as 297.15, that is 25ºC
    xf = xi+dg
    Dbc = -R*T*(math.log(xf)+((1-xf)/xf)*math.log(1-xf)) + R*T*(math.log(xi)+((1-xi)/xi)*math.log(1-xi))
    return Dbc

Again, these formulae are used here:

_ImpactScore_1_ = $\Delta x$; <br>
Computing for the initial ore-grade:
$x_i=e^{\alpha}(\frac{A}{T}-1)^{\beta}$ <br>
$x_f=x_i-\Delta x$ <br>

$b_c(x)=-RTº[ln(x)+(1-x)ln(1-x)]$ <br>
$\Delta b_c = CF_2 = |{b_c(x_i-\Delta g)-b_c{x_i}}|$, unit: kJ <br>
R = 8.314 kJ/kmol•K <br>
Tº = 290.15K = 17ºC <br>

In [67]:
# Step 1: Calculates the initial concentration of each mine and add it to OGD_df 
xi_list = []
for index,row in OGD_df.iterrows():
    xi = OG_ini(row['alpha'],row['beta'],row['URR'],row['CME'])
    xi_list.append(xi)
OGD_df['Initial Concentration'] = xi_list

# Step 2: For the rows in combined_IS_df that matches the element in OGD_df, add its xi
for index,row in combined_IS_df.iterrows():
    xi = OGD_df[OGD_df['Metal']==row['flow name']]['Initial Concentration']
    combined_IS_df.at[index,'Initial Concentration'] = xi.iloc[0] # the .iloc[0] is to convert xi from a Series to a float that can be easily read by pandas

# Step 3: For the rows in combined_IS_df that matches the element in OGD_df, add its k
for index,row in combined_IS_df.iterrows():
    xi = OGD_df[OGD_df['Metal']==row['flow name']]['k']
    combined_IS_df.at[index,'k'] = xi.iloc[0] # the .iloc[0] is to convert k from a Series to a float that can be easily read by pandas

# Step 3: Add the symbols to the DataFrame
combined_IS_df['symbol'] = ['Al','Cr','Cu','Fe','Pb','Mo','Ni','Zn']

# Step 4: Add the molar mass of each element correspondingly
M_list = []
for index,row in combined_IS_df.iterrows():
    M = element(row['symbol']).atomic_weight
    M_list.append(M)
combined_IS_df['M'] = M_list

## Option 1
# Step 5: Compute for the delta_bc value 
delta_bc_list = []
for index,row in combined_IS_df.iterrows():
    dbc = Delta_bc(row['Initial Concentration']/100,row['OG decrease']/100)
    dbc_m = dbc/row['M']
    delta_bc_list.append(dbc_m)
combined_IS_df['CF2_bc'] = delta_bc_list

## Option 2
# Step 5: Compute for the delta_bc value 
ERC_list = []
for index,row in combined_IS_df.iterrows():
    ERC = row['k']*Delta_bc(row['Initial Concentration']/100,row['OG decrease']/100)
    ERC_m = ERC/row['M']
    ERC_list.append(ERC_m)
combined_IS_df['CF2_ERC'] = ERC_list

combined_IS_df

,flow name,OG decrease,Initial Concentration,k,symbol,M,CF2_bc,CF2_ERC
0,Aluminium,-4.825921e-10,0.667892,1041.0,Al,26.981538,6.481779e-08,6.747532e-05
1,Chromium,-2.061579e-09,0.529672,18.0,Cr,51.996100,1.810542e-07,3.258975e-06
2,Copper,-4.125942e-09,0.037055,170.0,Cu,63.546000,4.227735e-06,7.187150e-04
3,Iron,-1.820008e-10,1.117443,78.0,Fe,55.845000,7.075183e-09,5.518643e-07
4,Lead,-1.391878e-09,0.121571,21.0,Pb,207.200000,1.333763e-07,2.800902e-06
5,Molybdenum,-5.935842e-10,0.003906,660.0,Mo,95.950000,3.820776e-06,2.521712e-03
6,Nickel,-1.098175e-09,0.031114,136.0,Ni,58.693400,1.450879e-06,1.973196e-04
7,Zinc,-1.759550e-09,0.317224,13.0,Zn,65.380000,2.049806e-07,2.664748e-06


### 2.3 CF2_dict construction and unique CF2 assignment (preparation for the final calculation in BW)
This step will take a few minutes

#### Choice of $CF_2$
CF2 of $\Delta b_c$: columns 4 & 6 <br>
CF2 of $\Delta ERC$: columns 4 & 7

In [69]:
# Convert the new df into a dictionary for the application of CF2
CF2_dict = dict(zip(combined_IS_df.iloc[:,4],combined_IS_df.iloc[:,7]))

CF2_dict

{'Al': 6.747532080109114e-05,
 'Cr': 3.258974993435593e-06,
 'Cu': 0.0007187149763151399,
 'Fe': 5.518642846788093e-07,
 'Pb': 2.8009021293286964e-06,
 'Mo': 0.0025217123603539994,
 'Ni': 0.00019731955385325205,
 'Zn': 2.6647481967383955e-06}

In [70]:
# we need a if-else filter here to see if the flow is elementary. If yes, do not go through the cf_calculator

method_data = []
extra_info = []

not_found = []
not_emission = [] # can be used to check for the flows that aren't considered

i=0
for flow in biosphere:
    cf = None
    parts = None #reset parts every time
    # Only consider the elementary flows going into the biosphere
    # Find flows that contain any of the search terms
    if isinstance(flow.get('categories'), tuple) and len(flow['categories']) > 0 and flow['unit'].lower() == 'kilogram' and flow['categories'][0].lower() == 'natural resource':

        flow_name_cleaned = clean_chemical_name(flow['name'])

        sleep(0.3) #Slow down the request to the API to avoid sudden shut down of connection

        max_retries = 3
        for attempt in range(max_retries):
            try:
                compound = pcp.get_compounds(flow_name_cleaned,'name')
                if compound: 
                    c = compound[0]
                    mf = c.molecular_formula
                    mw = c.molecular_weight
            
                    parts = chem_comp(mf)

                    # If the flow contains only one element, we can directly use its CF value
                    if len(parts)==1:
                        if has_elem(parts, CF2_dict): #if the only element is a part of the metal group
                            cf = CF2_dict.get(parts[0][0])   
                            print(f"The cf of {mf} is decided to be {cf}")
                        else:
                            print(f"{flow['name']} does not contain any element of the scope.")       
                    # If the flow contains a compound, use the cf_calculator
                    elif len(parts)>1:
                        if has_elem(parts,CF2_dict):
                            cf = cf_calculator(CF2_dict,mw,parts)
                            print(f"The cf of {mf} is calculated to be {cf}")
                        else:
                            print(f"{flow['name']} does not contain any element of the scope.")   
                    else: 
                        print(f"The molecular compound does not contain any element.")
                    
                    # Combine the original flow name, 
                    if cf is not None:
                        method_data.append((flow.key,float(cf)))
                        extra_info.append([flow.key,flow['name'],flow_name_cleaned,mf,mw,float(cf)])
                    else: 
                        extra_info.append([flow.key,flow['name'],flow_name_cleaned,mf,mw,None])
                        print('This compound does not contain any element of the scope.')
                else:
                    print(f"Compound not found for: {flow['name']}")
                    not_found.append([flow.key,flow['name']])
                break

            except PubChemHTTPError as e:
                # Check if it's a 502 or another server error
                if "502" in str(e) and attempt < max_retries - 1:
                    print(f"Server hit a 502 for {flow_name_cleaned}. Retrying in 2 seconds... (Attempt {attempt + 1}/{max_retries})")
                    time.sleep(2)  # Give the server a moment to recover
                else:
                    print(f"Failed to fetch {flow_name_cleaned} after multiple attempts or met a different error: {e}")
                    # Handle or log the permanent failure here
                    break
    else:
        not_emission.append([flow.key,flow['name']])


Ag is not included in the scope.
Silver, Ag 1.8E-6%, in mixed ore, in ground does not contain any element of the scope.
This compound does not contain any element of the scope.
Br is not included in the scope.
Bromine, 0.23% in water does not contain any element of the scope.
This compound does not contain any element of the scope.
Xe is not included in the scope.
Xenon, in air does not contain any element of the scope.
This compound does not contain any element of the scope.
Ag is not included in the scope.
Silver, Ag 2.1E-4%, Au 2.1E-4%, in ore, in ground does not contain any element of the scope.
This compound does not contain any element of the scope.
S is not included in the scope.
Sulfur, in ground does not contain any element of the scope.
This compound does not contain any element of the scope.
The cf of Cu is decided to be 0.0007187149763151399
C is not included in the scope.
H is not included in the scope.
F is not included in the scope.
N is not included in the scope.
O is n

In [ ]:
# Again, activate this block of code when you want to know what went on in the previous step

#PC_df = pd.DataFrame(extra_info)
#NF_df = pd.DataFrame(not_found)
#method_df = pd.DataFrame(method_data)
#method_df.to_csv("data/PubChem_CF2_Cu_Cathode_EI34.csv", index=False)
#PC_df.to_csv("data/PubChem_CF2_Cu_Cathode_EI34.csv", index=False)
#NF_df.to_csv("data/PubChem_CF2_Cu_Cathode_NotFound_EI34.csv", index=False)

### 2.4 Upload the new unique method file to BW
To get the final impact score, go to the Activity browser and work from there

In [71]:
# 1. Define the method name as a tuple for the desired hierarchy
method_name_tuple = (
   "Future Effort method",
    "Surplus ERC",
    "For Copper Cathode Production"  # Updated name
)

# 2. Define metadata for the method, including the unit and a description
method_metadata = {
    'unit': 'KJ-Eq',
    'description': 'impact analysis specifically for 500,000 kg of copper cathode production',
    'source': 'The values are taken from the appendix of ReCiPe 2016: https://www.rivm.nl/bibliotheek/rapporten/2016-0104.pdf',
    'version': '2.0', #I can try to update this number
    'num_cfs': len(method_data),
    'application': 'Input product-system metals characterization'
}

# 3. Create or load the Brightway2 Method object
method_object = bd.Method(method_name_tuple)

# 4. Forcefully register/write the method
try:
    # This will overwrite existing method
    method_object.register(**method_metadata) 
    method_object.write(method_data)
    
    print(f"\n✅ Successfully {'overwrote' if method_name_tuple in bd.methods else 'created'} method: {method_name_tuple}")
    print(f"   - Unit: {method_metadata['unit']}")
    print(f"   - Number of CFs: {len(method_data)}")
    
except Exception as e:
    print(f"\n❌ Failed to write method: {str(e)}")
    raise


# --- Verification ---
print("\n--- Verification ---")
if method_name_tuple in bd.methods:
    retrieved_method = bd.Method(method_name_tuple)
    loaded_data = retrieved_method.load()
    
    print(f"🔍 Method verification:")
    print(f"   Name: {retrieved_method.name}")
    print(f"   Metadata version: {retrieved_method.metadata.get('version', 'N/A')}")
    print(f"   Number of CFs loaded: {len(loaded_data)}")
    
    # Check for potential data loss
    if len(loaded_data) != len(method_data):
        print(f"⚠️  Warning: CF count mismatch. Expected {len(method_data)}, got {len(loaded_data)}")
    else:
        print("✅ CF count matches expected value")
        
    # Show sample CFs
    print("\nSample characterization factors (first 3):")
    for cf in loaded_data[:3]:
        flow = bd.get_activity(cf[0])
        print(f"   - {flow['name']}: {cf[1]} MJ-Eq")
else:
    print(f"❌ Error: Method {method_name_tuple} not found after writing attempt")

print("\nProcess completed")


✅ Successfully overwrote method: ('Future Effort method', 'Surplus ERC', 'For Copper Cathode Production')
   - Unit: KJ-Eq
   - Number of CFs: 45

--- Verification ---
🔍 Method verification:
   Name: ('Future Effort method', 'Surplus ERC', 'For Copper Cathode Production')
   Metadata version: 2.0
   Number of CFs loaded: 45
✅ CF count matches expected value

Sample characterization factors (first 3):
   - Copper, 2.19% in sulfide, Cu 1.83% and Mo 8.2E-3% in crude ore, in ground: 0.0007187149763151399 MJ-Eq
   - Lead, 5.0% in sulfide, Pb 3.0%, Zn, Ag, Cd, In, in ground: 2.8009021293286964e-06 MJ-Eq
   - Ni, Ni 3.7E-2%, Pt 4.8E-4%, Pd 2.0E-4%, Rh 2.4E-5%, Cu 5.2E-2% in ore, in ground: 0.00019731955385325205 MJ-Eq

Process completed
